# Drishti — Phase-0 Feasibility Spike (run on Google Colab)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

This notebook verifies every load-bearing assumption of the project by actually running it:
1. VizWiz data streams from Hugging Face (`lmms-lab/VizWiz-VQA`)
2. Candidate VLMs (Moondream-2, SmolVLM) answer blind-user photos — quality + latency
3. Surya OCR reads a real medicine strip photo (upload your own!)
4. IndicTrans2 (200M distilled) translates answers to Marathi/Hindi
5. MMS-TTS speaks Marathi/Hindi fully offline-capable models

**Fill in the findings log at the bottom when done** — those numbers go in the Sem-7 report.

---

### Troubleshooting

**If a cell already ran with the wrong transformers version** (e.g. you hit
`AttributeError: 'HfMoondream' object has no attribute 'all_tied_weights_keys'`):
the install cell pins `transformers<5`, but a version already imported into the kernel
isn't replaced by a pip downgrade. Do `Runtime → Restart session`, then `Runtime → Run all`.

**If Moondream still refuses to load**, don't lose the session to it — SmolVLM (§2b) uses
native transformers classes and is unaffected. Note "Moondream: failed to load" in the
findings table and carry SmolVLM forward as the baseline VLM; both are legitimate
candidates and the comparison is the point, not any one model.

In [ ]:
# transformers is pinned to 4.x on purpose. v5 requires models to call post_init()
# (which creates `all_tied_weights_keys`); Moondream-2's trust_remote_code file predates
# that and fails to load on v5 with:
#   AttributeError: 'HfMoondream' object has no attribute 'all_tied_weights_keys'
# IndicTrans2 (also trust_remote_code) is at risk of the same issue. Revisit the pin once
# upstream updates -- if you unpin, expect to swap Moondream for a natively-supported VLM.
%pip install -q "transformers<5" accelerate bitsandbytes datasets timm einops soundfile
import torch, transformers
print('transformers:', transformers.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Stream sample VizWiz images (real blind-user photos)

In [ ]:
from datasets import load_dataset
from itertools import islice
import matplotlib.pyplot as plt

stream = load_dataset('lmms-lab/VizWiz-VQA', split='val', streaming=True)
samples = list(islice(stream, 6))
print('columns:', samples[0].keys())

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, s in zip(axes.flat, samples):
    ax.imshow(s['image'])
    ax.set_title(s['question'][:60], fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2a. VLM candidate: Moondream-2 (~1.9B)

In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

md_model = AutoModelForCausalLM.from_pretrained(
    'vikhyatk/moondream2', trust_remote_code=True, torch_dtype=torch.float16, device_map=DEVICE)

def moondream_answer(img, question):
    try:  # newer revisions
        return md_model.query(img, question)['answer']
    except AttributeError:  # older revisions
        tok = AutoTokenizer.from_pretrained('vikhyatk/moondream2', trust_remote_code=True)
        enc = md_model.encode_image(img)
        return md_model.answer_question(enc, question, tok)

for s in samples[:4]:
    t0 = time.time()
    ans = moondream_answer(s['image'], s['question'])
    print(f"[{time.time()-t0:.1f}s] Q: {s['question'][:60]}\n         A: {ans}\n")

## 2b. VLM candidate: SmolVLM-Instruct (~2B)

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq

sv_proc = AutoProcessor.from_pretrained('HuggingFaceTB/SmolVLM-Instruct')
sv_model = AutoModelForVision2Seq.from_pretrained(
    'HuggingFaceTB/SmolVLM-Instruct', torch_dtype=torch.float16, device_map=DEVICE)

def smolvlm_answer(img, question):
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': question}]}]
    prompt = sv_proc.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = sv_proc(text=prompt, images=[img], return_tensors='pt').to(DEVICE)
    out = sv_model.generate(**inputs, max_new_tokens=64)
    text = sv_proc.batch_decode(out, skip_special_tokens=True)[0]
    return text.split('Assistant:')[-1].strip()

for s in samples[:4]:
    t0 = time.time()
    ans = smolvlm_answer(s['image'], s['question'])
    print(f"[{time.time()-t0:.1f}s] Q: {s['question'][:60]}\n         A: {ans}\n")

## 3. OCR spike: PaddleOCR on a medicine strip

Upload a photo of a real medicine strip (back side, where name + expiry are printed).
If no upload, it falls back to a VizWiz sample.

**Why PaddleOCR and not Surya.** The spike originally targeted Surya, which was dropped after
it proved unusable for this project:

- Surya **2.x** moved recognition behind an inference *server* (vllm via Docker, or
  llama.cpp). It fails in Colab with `SpawnError: docker binary not found`, and a phone app
  that needs a separate inference server running alongside it defeats the entire offline
  on-device premise of Drishti.
- Surya **0.x** runs in-process, but its call signature churned across patch releases
  (`det_predictor` accepted, then silently dropped by 0.22), costing several debug cycles.

PaddleOCR is the better fit on the axes that actually matter here: a stable API, `lang=
'devanagari'` covering **Hindi *and* Marathi**, and mobile-optimized PP-OCR models built for
exactly the Android target in milestone M5–6.

We install the **CPU** build of paddlepaddle deliberately — it sidesteps CUDA-version
mismatch in Colab, and CPU timings are the honest measurement for a phone-bound app anyway.

In [ ]:
# CPU build on purpose: avoids CUDA-version mismatch in Colab, and CPU latency is the
# honest number for a phone-bound app. Restart is not needed -- these are fresh packages.
%pip install -q paddlepaddle paddleocr

import time
from importlib.metadata import version
from PIL import Image
import numpy as np

print('paddleocr version:', version('paddleocr'))

# --- upload image(s); falls back to a VizWiz sample if you skip the upload ---
try:
    from google.colab import files
    up = files.upload()
    strip_imgs = [Image.open(name).convert('RGB') for name in up] or [samples[0]['image']]
except Exception:
    strip_imgs = [samples[0]['image']]
print(f'{len(strip_imgs)} image(s) to OCR')

from paddleocr import PaddleOCR


def extract_lines(result):
    """Normalize PaddleOCR output to [(confidence, text)].

    Handles both result shapes so this cell survives a version bump:
      3.x .predict() -> objects/dicts carrying `rec_texts` + `rec_scores`
      2.x .ocr()     -> nested [[bbox, (text, score)], ...]
    """
    lines = []
    for page in result or []:
        texts = scores = None

        if isinstance(page, dict):
            texts, scores = page.get('rec_texts'), page.get('rec_scores')
        else:
            texts, scores = getattr(page, 'rec_texts', None), getattr(page, 'rec_scores', None)
            if texts is None and hasattr(page, 'json'):
                blob = page.json
                blob = blob.get('res', blob) if isinstance(blob, dict) else {}
                texts, scores = blob.get('rec_texts'), blob.get('rec_scores')

        if texts is not None:
            scores = scores if scores is not None else [float('nan')] * len(texts)
            lines.extend(zip(scores, texts))
            continue

        if isinstance(page, list):          # 2.x layout
            for item in page:
                try:
                    _bbox, (text, score) = item
                    lines.append((score, text))
                except (TypeError, ValueError):
                    continue
    return lines


def run_paddle(images, lang):
    """Run OCR, preferring the 3.x predict() API and falling back to 2.x ocr()."""
    ocr = PaddleOCR(lang=lang)
    arrays = [np.array(img) for img in images]

    out = []
    t0 = time.time()
    for arr in arrays:
        if hasattr(ocr, 'predict'):
            out.extend(extract_lines(ocr.predict(arr)))
        else:
            out.extend(extract_lines(ocr.ocr(arr)))
    return out, time.time() - t0


# --- English pass: Indian medicine strips print drug name / EXP / MRP in Latin script ---
en_lines, en_secs = run_paddle(strip_imgs, 'en')
print(f'\n=== lang="en" — {en_secs:.1f}s, {len(en_lines)} lines ===')
for conf, text in en_lines:
    print(f'  {conf:.2f}  {text}')

# --- Devanagari pass: load-bearing for Read mode (Marathi/Hindi labels and signage) ---
try:
    dev_lines, dev_secs = run_paddle(strip_imgs, 'devanagari')
    print(f'\n=== lang="devanagari" — {dev_secs:.1f}s, {len(dev_lines)} lines ===')
    for conf, text in dev_lines:
        print(f'  {conf:.2f}  {text}')
except Exception as e:
    dev_lines = []
    print(f'\ndevanagari model unavailable: {e}')

# --- downstream cells consume ocr_text; keep the English pass as the medicine-mode input ---
ocr_lines = en_lines
ocr_text = ' '.join(text for _, text in ocr_lines)
print(f'\nocr_text ({len(ocr_lines)} lines): {ocr_text[:200]}')

In [ ]:
# Expiry-date extraction spike: regex over OCR text (medicine-mode building block).
# These same patterns now live in app/parsers.py -- keep them in sync if you tune them here.
import re

date_pat = re.compile(r'(?:EXP|Expiry|Exp\.?)[:\s.]*([A-Z]{3}[.\s/-]?\d{2,4}|\d{1,2}[./-]\d{2,4})', re.I)
mrp_pat = re.compile(r'(?:MRP|Rs\.?|₹)[:\s.]*([\d,.]+)', re.I)

print('OCR text  :', ocr_text[:300])
print('expiry candidates:', date_pat.findall(ocr_text))
print('MRP candidates   :', mrp_pat.findall(ocr_text))

## 4. Translation: IndicTrans2 distilled 200M (en → Marathi/Hindi)

In [ ]:
%pip install -q IndicTransToolkit
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

it_name = 'ai4bharat/indictrans2-en-indic-dist-200M'
it_tok = AutoTokenizer.from_pretrained(it_name, trust_remote_code=True)
it_model = AutoModelForSeq2SeqLM.from_pretrained(it_name, trust_remote_code=True).to(DEVICE)
ip = IndicProcessor(inference=True)

def translate(sentences, tgt='mar_Deva'):
    batch = ip.preprocess_batch(sentences, src_lang='eng_Latn', tgt_lang=tgt)
    inputs = it_tok(batch, truncation=True, padding='longest', return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = it_model.generate(**inputs, max_length=128, num_beams=4)
    dec = it_tok.batch_decode(out, skip_special_tokens=True)
    return ip.postprocess_batch(dec, lang=tgt)

tests = ['This medicine expires in March 2027.',
         'This is a five hundred rupee note.',
         'There is a chair to your left and a doorway ahead.']
t0 = time.time()
print('Marathi:', *translate(tests, 'mar_Deva'), sep='\n  ')
print('Hindi  :', *translate(tests, 'hin_Deva'), sep='\n  ')
print(f'({time.time()-t0:.1f}s for 6 translations)')

## 5. Offline TTS: MMS-TTS Marathi / Hindi

In [ ]:
from transformers import VitsModel, AutoTokenizer as TTSTok
from IPython.display import Audio, display

for lang, text in [('mar', translate(['This medicine expires in March 2027.'], 'mar_Deva')[0]),
                   ('hin', translate(['This medicine expires in March 2027.'], 'hin_Deva')[0])]:
    tts = VitsModel.from_pretrained(f'facebook/mms-tts-{lang}')
    ttok = TTSTok.from_pretrained(f'facebook/mms-tts-{lang}')
    inputs = ttok(text, return_tensors='pt')
    with torch.no_grad():
        wav = tts(**inputs).waveform
    print(lang, '→', text)
    display(Audio(wav.numpy(), rate=tts.config.sampling_rate))

## 6. Findings log (fill this in — goes into the Sem-7 report)

| Component | Works? | Latency | Quality notes |
|---|---|---|---|
| VizWiz streaming (HF) | | | |
| Moondream-2 | | s/answer | |
| SmolVLM | | s/answer | |
| PaddleOCR `lang="en"` on strip | | s (CPU) | drug name readable? EXP? MRP? |
| PaddleOCR `lang="devanagari"` | | s (CPU) | Marathi/Hindi text readable? |
| Expiry/MRP regex | | | |
| IndicTrans2 en→mr/hi | | | translation quality (ask a native speaker) |
| MMS-TTS mr/hi | | | voice intelligibility |

**Decisions to make after this run:**
1. Which VLM to carry forward as the fine-tuning base?
2. Is PaddleOCR accurate enough on strips, or does medicine mode need a fine-tuned recognizer?
3. Next notebook: `01_vizwiz_baseline.ipynb` — the official baseline number.

---

### Findings already banked from this spike (write these into the report)

**1. VLM hallucination is a live safety risk, not a hypothetical.** Asked "what is this
medicine?", Moondream-2 answered with a confident, fabricated drug classification *and*
invented an ingredient list. This is direct evidence for the medicine-mode guardrail in
`app/drug_db.py`: a drug name is reported only when OCR text matches a verified database,
never from VLM generation. Screenshot that output — it justifies the safety architecture
in one slide.

**2. Verbosity is a metric problem, not just a UX one.** VizWiz scores by exact match
against short crowd answers, so a verbose-but-correct answer scores ~0. This directly
motivates the `PROMPT_SUFFIX` constraint in notebook 01 and the terse-answer preference
when selecting the base model. Measured here: SmolVLM answered ~2.5× faster than Moondream-2
*and* more tersely — better on both axes that matter for this project.

**3. Deployment architecture is a selection criterion, not an afterthought.** The OCR engine
was switched from Surya to PaddleOCR mid-spike: Surya 2.x requires a separate inference
server (vllm/Docker or llama.cpp), which is incompatible with offline on-device operation,
and Surya 0.x churned its call signature across patch releases. "Runs in-process, offline,
stable API" is now an explicit requirement for every component in this project — the same
filter that later favours PP-OCR mobile models for the Android port.

**4. Reproducibility needs explicit pins.** Two separate breakages came from unpinned
dependencies resolving to newer majors (`transformers` v5 breaking `trust_remote_code`
models; `surya-ocr` 2.x rearchitecting). Both notebooks now pin deliberately and comment
why. In a fast-moving small-model ecosystem, an unpinned environment is not reproducible —
worth a paragraph in the report's methodology section.